# 🐝 ME2 CHAT-SWARM — внешний GPU-узел — phi4:14b (Google Colab)

Этот ноутбук превращает бесплатную GPU-сессию Colab в вычислительный узел роя ME2 CHAT-SWARM:

1. Ставит **Ollama** + модель **llama3.1:8b** (можно заменить в ячейке на qwen2.5:7b / llama3.2:3b).
2. Открывает её наружу через бесплатный туннель **cloudflared** (trycloudflare.com, без аккаунта и ключей).
3. Регистрирует узел у роя: `POST {PUBLIC_BASE}/colab/register?XTransformPort=3046`.

После регистрации рой сам:
- ставит узел **первым** в LLM-цепочке (груз GPU уходит с квотных провайдеров);
- поднимает rate-governor с 15 до **30 вызовов/мин** на весь рой;
- зондирует узел каждые 5 мин: 3 зонда подряд мимо → автоматическое исключение (рой живёт дальше).

⚠️ Сеанс Colab free живёт ориентировочно 4–12 часов. При перезапуске ноутбука туннель даёт **НОВЫЙ** URL — просто повторите ячейку регистрации (шаг 3).

⚡ Runtime → Change runtime type → **T4 GPU** (желательно, но CPU-сеанс тоже работает с меньшей моделью).

**ВАРИАНТ: `phi4:14b`** — Microsoft Phi-4 14B (tier 75), ~9.1 GB VRAM. Имя узла: `colab-4-phi4-14b`.


## Шаг 1. Установка Ollama и запуск сервера

In [ ]:
# Установка Ollama (официальный скрипт)
!curl -fsSL https://ollama.com/install.sh | sh
print('\u2705 Ollama установлен')

In [ ]:
# Запуск ollama serve в фоне + ожидание готовности порта 11434
import subprocess, time, socket, os

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'   # слушаем все интерфейсы — через них придёт туннель
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'        # модель всегда в VRAM — нет задержек на reload
os.environ['OLLAMA_ORIGINS'] = '*'            # CORS для свободных зондов роя

srv = subprocess.Popen(['ollama', 'serve'],
                       stdout=open('ollama_serve.log', 'w'),
                       stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        s = socket.create_connection(('127.0.0.1', 11434), timeout=1); s.close()
        print('\u2705 Ollama сервер поднят на :11434'); break
    except OSError:
        time.sleep(1)
else:
    print('\u274c не дождались порта — смотрим ollama_serve.log:')
    print(open('ollama_serve.log').read()[-2000:])

## Шаг 2. Загрузка модели и локалный самотест

In [ ]:
# Модель роя по умолчанию. На T4 загрузка ~2-4 мин.
# Альтернативы: 'qwen2.5:7b' (CPU/T4), 'llama3.2:3b' (маленькая), 'qwen2.5:14b' (мощнее, медленнее)
SWARM_MODEL = 'phi4:14b'
NODE_NAME = 'colab-4-phi4-14b'
!ollama pull $SWARM_MODEL
!ollama list

In [ ]:
# Локальный самотест: один чат-вызов точно как их будет делать рой
import json, urllib.request, time

def swarm_chat(prompt, model=SWARM_MODEL):
    body = json.dumps({
        'model': model, 'stream': False, 'keep_alive': -1,
        'messages': [
            {'role': 'system', 'content': 'Ты — агент роя ME2 CHAT-SWARM. Отвечай кратко.'},
            {'role': 'user', 'content': prompt},
        ],
        'options': {'num_predict': 500, 'temperature': 0.7},
    }).encode()
    t0 = time.time()
    req = urllib.request.Request('http://127.0.0.1:11434/api/chat', data=body,
                                 headers={'Content-Type': 'application/json'})
    d = json.loads(urllib.request.urlopen(req, timeout=300).read())
    return d['message']['content'], time.time() - t0

reply, dt = swarm_chat('Самотест: скажи одним словом, что ты живой.')
print(f'\u2705 ответ за {dt:.1f}с: {reply[:200]}')

## Шаг 3. Туннель cloudflared и РЕГИСТРАЦИЯ узла у роя

In [ ]:
# Установка cloudflared и запуск бесплатного quick-туннеля (без аккаунта Cloudflare)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import subprocess, re, time
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
                       stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)

TUNNEL_URL = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        TUNNEL_URL = m.group(0); break

if TUNNEL_URL:
    print('TUNNEL_URL =', TUNNEL_URL)
else:
    print('\u274c туннель не поднялся — хвост tunnel.log:')
    print(open('tunnel.log').read()[-1500:])

In [ ]:
# РЕГИСТРАЦИЯ узла у роя.
# Вставьте публичный адрес песочницы (кнопка «Open in New Tab» над превью,
# без хвостового слэша), например: 'https://xxxxx.z.ai'
PUBLIC_BASE = 'PASTE_PUBLIC_URL_HERE'

import json, urllib.request

def register_node(public_base, tunnel_url, model=SWARM_MODEL, node_name=NODE_NAME):
    body = json.dumps({'url': tunnel_url, 'model': model, 'name': node_name}).encode()
    req = urllib.request.Request(
        f'{public_base.rstrip("/")}/colab/register?XTransformPort=3046',
        data=body, headers={'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req, timeout=60).read())

if TUNNEL_URL and not PUBLIC_BASE.startswith('PASTE'):
    resp = register_node(PUBLIC_BASE, TUNNEL_URL)
    print(json.dumps(resp, ensure_ascii=False, indent=2))
    print('\n\ud83d? УЗЕЛ ПОДКЛЮЧЁН — рой уже работает через этот GPU' if resp.get('ok') else '\n\u274c регистрация не прошла, см. detail выше')
else:
    print('\u26a0️ Ячейка в режиме ожидания: вставьте PUBLIC_BASE и повторите.')
    print(f'\nЗапомните URL узла для ручной регистрации:\nTUNNEL_URL = {TUNNEL_URL}\n\nЕсли не получилось зарегистрировать отсюда — передайте этот URL оператору/инженеру роя, он выполнит:\ncurl -X POST "<адрес песочницы>/colab/register?XTransformPort=3046" -H "Content-Type: application/json" -d \'{{"url": "{TUNNEL_URL}", "model": "{SWARM_MODEL}"}}\'')

## Шаг 4. Поддержание жизни сеанса

- Оставьте вкладку Colab открытой; сворачивание — ок, закрытие вкладки надолго → сеанс умрёт быстрее.
- Рой сам зондирует узел каждые 5 мин; смерть узла не останавливает рой.
- После перезапуска сеанса — запустите ячейки заново и повторите регистрацию (шаг 3).

**Проверка со стороны роя:**
- `GET {PUBLIC_BASE}/colab/status?XTransformPort=3046` — состояние узла;
- `GET {PUBLIC_BASE}/health?XTransformPort=3046` — `llmProviders[0].name == "colab-node"`, `governor.colabBoost == true`.